# BERTopic: Temporal Topic Analysis (Native)

Uses BERTopic's native `topics_over_time()` with evolution & global tuning.
Topic -1 (outliers) excluded.

In [1]:
import time
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from bertopic import BERTopic
import warnings
warnings.filterwarnings("ignore")

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
BASE_DIR = Path("../../../../data/preprocess")
MODEL_DIR = Path("../../../../models/bertopic/tuning/hdbscan")
RESULT_DIR = Path("../../../../results/bertopic/temporal")
VERSION = "v1"
TOP_N_WORDS = 10
RBO_P = 0.9

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

In [3]:
def rbo(list_1, list_2, p=0.9):
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    score = 0.0
    for d in range(1, k + 1):
        agreement = len(set(list_1[:d]) & set(list_2[:d])) / d
        score += (p ** (d - 1)) * agreement
    return score * (1 - p)


def calculate_irbo(topics_words_list, p=0.9):
    if len(topics_words_list) < 2:
        return 0.0
    scores = [1.0 - rbo(topics_words_list[i], topics_words_list[j], p)
              for i, j in combinations(range(len(topics_words_list)), 2)]
    return np.mean(scores)

## Load Models & Data + topics_over_time

In [4]:
all_models = {}
all_data = {}
all_years = {}
all_tot = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")

    model = BERTopic.load(str(MODEL_DIR / f"best_{subject}_quality"))
    all_models[subject] = model

    df = pd.read_csv(BASE_DIR / subject / "emb" / f"{VERSION}.csv")
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    df["topic"] = model.topics_

    # Native topics_over_time
    print(f"  Computing topics_over_time...")
    start = time.time()
    tot_df = model.topics_over_time(
        docs=df["text"].tolist(),
        timestamps=df["year"].tolist(),
        evolution_tuning=True, global_tuning=True,
    )
    elapsed = time.time() - start
    all_tot[subject] = tot_df

    n_outliers = (df["topic"] == -1).sum()
    df = df[df["topic"] != -1].reset_index(drop=True)
    all_data[subject] = df
    years = sorted(df["year"].unique())
    all_years[subject] = years

    n_topics = len(set(model.topics_)) - (1 if -1 in model.topics_ else 0)
    print(f"  {subject}: {len(df):,} docs (excl. {n_outliers:,} outliers), "
          f"{n_topics} topics, {len(years)} years [{elapsed:.1f}s]")

print(f"\n✅ All subjects loaded")


Loading cs...
  Computing topics_over_time...
  cs: 98,534 docs (excl. 67,222 outliers), 261 topics, 26 years [46.4s]

Loading math...
  Computing topics_over_time...
  math: 87,138 docs (excl. 69,947 outliers), 150 topics, 26 years [29.5s]

Loading physics...
  Computing topics_over_time...
  physics: 80,033 docs (excl. 66,278 outliers), 232 topics, 26 years [47.8s]

✅ All subjects loaded


## Topic Prevalence Over Time

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    all_topic_ids = sorted(df["topic"].unique())

    global_tw = {}
    for tid in all_topic_ids:
        info = model.get_topic(tid)
        global_tw[tid] = ", ".join([w for w, _ in info[:5]]) if info else ""

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        counts = year_df["topic"].value_counts()
        for tid in all_topic_ids:
            count = counts.get(tid, 0)
            rows.append({"subject": subject, "year": year, "topic_id": tid,
                         "doc_count": count, "total_docs_year": len(year_df),
                         "proportion": round(count / len(year_df), 6) if len(year_df) > 0 else 0,
                         "top_words": global_tw.get(tid, "")})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_prevalence.csv", index=False)
    print(f"  {subject.upper()}: Saved topic_prevalence.csv")

  CS: Saved topic_prevalence.csv
  MATH: Saved topic_prevalence.csv
  PHYSICS: Saved topic_prevalence.csv


## Topic Word Evolution (Native BERTopic)

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    tot_df = all_tot[subject]
    tot_filtered = tot_df[tot_df["Topic"] != -1].copy()

    topic_words_per_year = {}
    rows = []
    for _, row in tot_filtered.iterrows():
        year = int(row["Timestamp"].year) if hasattr(row["Timestamp"], "year") else int(row["Timestamp"])
        tid = int(row["Topic"])
        words_str = row["Words"]
        if isinstance(words_str, str):
            words = [w.strip() for w in words_str.split(", ")][:TOP_N_WORDS]
        else:
            words = list(words_str)[:TOP_N_WORDS]

        topic_words_per_year[(year, tid)] = words
        rows.append({"subject": subject, "year": year,
                     "topic_id": tid, "top_words": ", ".join(words),
                     "frequency": int(row["Frequency"])})

    all_topic_words_per_year[subject] = topic_words_per_year
    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "topic_word_evolution.csv", index=False)
    print(f"  {subject.upper()}: Saved topic_word_evolution.csv ({len(rows)} rows)")

  CS: Saved topic_word_evolution.csv (4328 rows)
  MATH: Saved topic_word_evolution.csv (3572 rows)
  PHYSICS: Saved topic_word_evolution.csv (5162 rows)


## Per-Year Coherence & IRBO

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    topic_words_per_year = all_topic_words_per_year[subject]
    n_topics = len(set(model.topics_)) - (1 if -1 in model.topics_ else 0)

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    rows = []
    for year in years:
        year_df = df[df["year"] == year]
        year_texts = [t.split() for t in year_df["text"].tolist()]
        dictionary = Dictionary(year_texts)

        active_topics = sorted(year_df["topic"].unique())
        year_tw = [topic_words_per_year[(year, t)] for t in active_topics
                   if (year, t) in topic_words_per_year
                   and len(topic_words_per_year[(year, t)]) >= 2]

        if year_tw:
            cm = CoherenceModel(topics=year_tw, texts=year_texts,
                                dictionary=dictionary, coherence='c_v', processes=1)
            coherence = cm.get_coherence()
        else:
            coherence = 0.0

        irbo = calculate_irbo(year_tw, p=RBO_P)
        quality = 2 * coherence * irbo / (coherence + irbo) if (coherence + irbo) > 0 else 0.0

        print(f"  {year}: {len(year_df):>6,} docs | {len(active_topics):>3} topics | "
              f"Q={quality:.4f} (C={coherence:.4f}, IRBO={irbo:.4f})")

        rows.append({"subject": subject, "year": year, "num_docs": len(year_df),
                     "num_topics_total": n_topics, "num_topics_active": len(active_topics),
                     "coherence_cv": round(coherence, 6), "irbo_mean": round(irbo, 6),
                     "topic_quality": round(quality, 6)})

    pd.DataFrame(rows).to_csv(RESULT_DIR / subject / "per_year_metrics.csv", index=False)
    print(f"  Saved: per_year_metrics.csv")


Per-Year Metrics: CS (261 topics)
  2000:    211 docs |  47 topics | Q=0.8468 (C=0.7346, IRBO=0.9994)
  2001:    224 docs |  63 topics | Q=0.8659 (C=0.7637, IRBO=0.9997)
  2002:    306 docs |  65 topics | Q=0.8410 (C=0.7258, IRBO=0.9997)
  2003:    384 docs |  74 topics | Q=0.8548 (C=0.7466, IRBO=0.9997)
  2004:    436 docs |  95 topics | Q=0.8496 (C=0.7387, IRBO=0.9997)
  2005:    495 docs |  87 topics | Q=0.8671 (C=0.7657, IRBO=0.9996)
  2006:    501 docs |  88 topics | Q=0.8384 (C=0.7219, IRBO=0.9995)
  2007:    503 docs |  82 topics | Q=0.8051 (C=0.6740, IRBO=0.9995)
  2008:    508 docs | 100 topics | Q=0.8341 (C=0.7157, IRBO=0.9995)
  2009:    528 docs | 108 topics | Q=0.8353 (C=0.7174, IRBO=0.9996)
  2010:    703 docs | 126 topics | Q=0.8097 (C=0.6804, IRBO=0.9997)
  2011:    854 docs | 144 topics | Q=0.8187 (C=0.6933, IRBO=0.9996)
  2012:  1,217 docs | 161 topics | Q=0.8176 (C=0.6917, IRBO=0.9996)
  2013:  1,413 docs | 178 topics | Q=0.7849 (C=0.6462, IRBO=0.9996)
  2014:  1,56

## Topic Trends

In [8]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    all_topic_ids = sorted(df["topic"].unique())

    rows = []
    for tid in all_topic_ids:
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean()
        late_mean = proportions[topic_years[-5:]].mean()

        if early_mean < 1e-6 and late_mean < 1e-6:
            trend_ratio = 1.0
        else:
            trend_ratio = late_mean / max(early_mean, 1e-4)

        trend_label = "GROWING" if trend_ratio > 2.0 else ("DECLINING" if trend_ratio < 0.5 else "STABLE")

        info = model.get_topic(tid)
        top_words = ", ".join([w for w, _ in info[:5]]) if info else ""

        rows.append({"subject": subject, "topic_id": tid, "top_words": top_words,
                     "total_docs": len(topic_df),
                     "first_year": year_counts.index.min(), "last_year": year_counts.index.max(),
                     "early_proportion": round(early_mean, 6),
                     "late_proportion": round(late_mean, 6),
                     "trend_ratio": round(trend_ratio, 4), "trend": trend_label})

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")

  CS: Growing=70, Stable=118, Declining=73
  MATH: Growing=31, Stable=82, Declining=37
  PHYSICS: Growing=46, Stable=114, Declining=72


## Top 5 Growing & Declining Topics

In [9]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df["trend"] == "GROWING"].sort_values("trend_ratio", ascending=False)
    declining = trends_df[trends_df["trend"] == "DECLINING"].sort_values("trend_ratio", ascending=True)

    print(f"\n  📈 TOP 5 GROWING:")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | {row['trend_ratio']:>7.2f}x | "
              f"{row['early_proportion']:.4f} → {row['late_proportion']:.4f} | {row['top_words']}")

    print(f"\n  📉 TOP 5 DECLINING:")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | {row['trend_ratio']:>7.4f}x | "
              f"{row['early_proportion']:.4f} → {row['late_proportion']:.4f} | {row['top_words']}")


  CS

  📈 TOP 5 GROWING:
    T 52 |   15.18x | 0.0004 → 0.0058 | reasoning, cot, llms, math, mathematical
    T 12 |    7.76x | 0.0016 → 0.0121 | driving, autonomous, traffic, vehicles, vehicle
    T  2 |    7.60x | 0.0031 → 0.0237 | visual, multimodal, vision, image, language
    T 18 |    7.49x | 0.0014 → 0.0101 | safety, attacks, attack, llms, jailbreak
    T 39 |    6.61x | 0.0012 → 0.0078 | weather, climate, forecasting, precipitation, air

  📉 TOP 5 DECLINING:
    T131 |  0.0176x | 0.0574 → 0.0010 | grid, hpc, computing, workflows, workflow
    T218 |  0.0232x | 0.0125 → 0.0003 | relay, relaying, channel, relays, forward
    T109 |  0.0421x | 0.0290 → 0.0012 | codes, ldpc, decoding, check, parity
    T231 |  0.0464x | 0.0104 → 0.0005 | mining, frequent, itemsets, association, pattern
    T  4 |  0.0467x | 0.1780 → 0.0083 | logic, calculus, type, semantics, programs

  MATH

  📈 TOP 5 GROWING:
    T 82 |   13.06x | 0.0006 → 0.0074 | neural, networks, deep, training, relu
    T 37

## Evolution Summary

In [10]:
for subject in LIST_SUBJECT:
    metrics_df = pd.read_csv(RESULT_DIR / subject / "per_year_metrics.csv")
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")
    n_topics = len(set(all_models[subject].topics_)) - (1 if -1 in all_models[subject].topics_ else 0)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])

    summary = {"subject": subject, "num_topics": n_topics,
               "num_years": len(metrics_df),
               "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
               "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
               "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
               "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
               "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
               "quality_std": round(metrics_df["topic_quality"].std(), 6),
               "topics_growing": g, "topics_stable": s, "topics_declining": d}

    pd.DataFrame([summary]).to_csv(RESULT_DIR / subject / "evolution_summary.csv", index=False)
    print(f"  {subject.upper()}: C={summary['coherence_mean']:.4f}, "
          f"IRBO={summary['irbo_mean']:.4f}, Q={summary['quality_mean']:.4f} | "
          f"↑{g} →{s} ↓{d}")

  CS: C=0.7068, IRBO=0.9995, Q=0.8274 | ↑70 →118 ↓73
  MATH: C=0.6576, IRBO=0.9992, Q=0.7925 | ↑31 →82 ↓37
  PHYSICS: C=0.6907, IRBO=0.9994, Q=0.8167 | ↑46 →114 ↓72
